# RA-EEM — Servidor API

**Instrucciones:**
1. Sube los 4 archivos Python a este Colab: `simulador_api.py`, `analitica.py`, `cuestionario.py`, `lanzar.py`
2. Ejecuta las celdas **en orden**
3. La celda final te da la URL pública — **cópiala al frontend**
4. Deja el Colab abierto mientras uses el frontend


In [ ]:
# ── CELDA 1: Instalar dependencias ──────────────────────────
!pip install flask flask-cors pyngrok --quiet

In [ ]:
# ── CELDA 2: Subir archivos Python ──────────────────────────
# Ejecuta esta celda y usa el botón 'Choose Files' para subir:
#   simulador_api.py, analitica.py, cuestionario.py
from google.colab import files
uploaded = files.upload()
print('Archivos subidos:', list(uploaded.keys()))

In [ ]:
# ── CELDA 3: Verificar archivos ─────────────────────────────
import os
required = ['simulador_api.py', 'analitica.py', 'cuestionario.py']
for f in required:
    status = '✅' if os.path.exists(f) else '❌ FALTA'
    print(f'{status}  {f}')

In [ ]:
# ── CELDA 4: Configurar ngrok ────────────────────────────────
# Obtén tu authtoken GRATIS en https://dashboard.ngrok.com/signup
# Crea cuenta, ve a 'Your Authtoken' y pégalo abajo:

NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUI'  # <── reemplazar

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok configurado')

In [ ]:
# ── CELDA 5: Lanzar servidor Flask ───────────────────────────
import threading
import json
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

from simulador_api import simular_api

app = Flask(__name__)
CORS(app)  # Permite requests desde GitHub Pages

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'RA-EEM v0.3.4'})

@app.route('/run', methods=['POST'])
def run_simulation():
    try:
        params = request.get_json(force=True)
        if not params:
            return jsonify({'error': 'No se recibieron parámetros'}), 400

        # Validaciones mínimas
        required_keys = ['_derived_agent_profile', '_alter_settings']
        for k in required_keys:
            if k not in params:
                return jsonify({'error': f'Falta clave requerida: {k}'}), 400

        result = simular_api(params)
        return jsonify(result)

    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500

# Lanzar Flask en thread separado
def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()

# Crear tunnel público
public_url = ngrok.connect(5000).public_url
print('\n' + '='*50)
print('🚀 SERVIDOR RA-EEM ACTIVO')
print('='*50)
print(f'URL pública: {public_url}')
print(f'Health check: {public_url}/health')
print('='*50)
print('\n⚠️  Copia esta URL al frontend (campo SERVER URL)')
print('⚠️  Mantén esta celda corriendo')

In [ ]:
# ── CELDA 6 (opcional): Test local ───────────────────────────
# Corre esto para verificar que el simulador funciona antes
# de conectar el frontend

test_params = {
    'lambda_s': 1.181, 'lambda_r': 0.6, 'lambda_g': 0.35, 'lambda_I': 0.05,
    'sigma': 1.0, 'kappa': 2.2, 'chi': 2.5, 'alpha': 0.029,
    'eta_symp': 0.0045, 'eta_evit': 0.003, 'beta': 0.066,
    'p_burnout': 1.5, 'q_fatigue': 3, 'alpha_a': 0.02, 'alpha_e': 0.01,
    'gamma_plus': 0.031, 'gamma_minus': 0.04, 'nu': 0.08,
    'mu_a': 0.2, 'delta_a': 0.08, 'mu_e': 0.1, 'delta_e': 0.05,
    'tau': 17, 'rho_e': 0.04, 'repair_threshold': 0.44,
    'repair_probability': 0.12, 'repair_strength': 0.8,
    'zeta': 0.045, 'noise_sigma': 0.061,
    '_derived_agent_profile': {
        'w_s': 0.381, 'w_a': 0.238, 'w_e': 0.286, 'w_d': 0.095,
        'A': 0.4, 'P_default': 0.785
    },
    '_alter_settings': {
        'context_type': 'FAMILIAL',
        'archetype': 'EMOTIONALLY_ABSENT',
        'custom_profile': {
            'w_s': 0.65, 'w_a': 0.8, 'w_e': 0.7, 'w_d': 0.4,
            'G': 0.5, 'A': 0.7, 'P_default': 0.75
        }
    }
}

result = simular_api(test_params)
print('✅ Simulación OK')
print('Atractor:', result['phenotype']['relational_stability']['attractor_type'])
print('Patrones:', result['phenotype']['emergent_patterns'])
print('Imagen generada:', result['image_base64'][:50] + '...')